# 🧬 Práctica B: Anotación de Genomas con conda en Google Colab

> ⚠️ **IMPORTANTE: antes de continuar**, lea la [guía de prácticas compartida](00_genome_annotation_common.md) — contiene el contexto biológico de cada caso, los conceptos clave, las plataformas y la descripción de los archivos de salida.

Esta práctica sigue el mismo flujo de trabajo que la **Práctica A** (Galaxy), pero ejecutando las herramientas desde la línea de comandos en un entorno Google Colab + conda. Esto le permite:

- Instalar y usar herramientas bioinformáticas de CLI sin necesitar un servidor institucional.
- Integrar los resultados directamente en Python con `pandas` y `matplotlib`.
- Comprender cómo se encadenan estas herramientas en un pipeline automatizado.

---

## ⚙️ Paso 0 — Configurar el entorno

Ejecute esta celda **primero**. Instala Miniconda y todas las herramientas necesarias.
⏱️ Puede tardar 8–12 minutos.

In [1]:
# Instalamos Conda para Colab

!pip install condacolab --quiet

import condacolab
condacolab.install() # Esto reinicia el kernel automáticamente

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 121.2 MB/s eta 0:00:00
✨🍰✨ Everything looks OK!
conda 24.11.2


Tras el reinicio del kernel, ejecuta esta celda para confirmar que el entorno quedó correctamente configurado y para dejar registradas las versiones de las herramientas base (útil para citar en la sección de métodos de informes/artículos).

In [ ]:
# Verificar instalación de condacolab y registrar versiones del entorno base
import os
import sys
import subprocess

os.environ['PATH'] = '/usr/local/bin:/opt/conda/bin:' + os.environ['PATH']

# Instalar biopython
!pip install biopython --quiet

def version_pip(paquete):
    res = subprocess.run(["pip", "show", paquete], capture_output=True, text=True)
    for linea in res.stdout.splitlines():
        if linea.startswith("Version:"):
            return linea.split(":", 1)[1].strip()
    return "⚠️ no encontrado"

conda_version = subprocess.run(["conda", "--version"], capture_output=True, text=True).stdout.strip()

# google-colab no se instala vía pip estándar en el entorno conda (es parte
# de la integración interna del runtime), por lo que 'pip show' no lo detecta
# de forma confiable. Verificamos disponibilidad importando el módulo.
try:
    import google.colab  # noqa: F401
    colab_estado = "disponible (ejecutando en Google Colab)"
except ImportError:
    colab_estado = "no detectado (¿no estás en Google Colab?)"

print("📦 Versiones del entorno base:")
print(f"  - Python: {sys.version.split()[0]}")
print(f"  - {conda_version}")
print(f"  - biopython: {version_pip('biopython')}")
print(f"  - Google Colab: {colab_estado}")

Una vez que Conda esté instalado, es posible que necesites reiniciar el entorno de ejecución de Colab (Runtime > Restart runtime) para que los cambios surtan efecto. Después de reiniciar, puedes ejecutar comandos `conda` en nuevas celdas de código.

### Instalar herramientas

Ya que hemos instalado **Conda** es necesario instalar las herramientas que vamos a usar para el ensamblaje.

In [2]:
# Instalar herramientas bioinformáticas

# Nota: antiSMASH NO se instala aquí porque entra en conflicto de versión de
# Python con bakta. Se instala por separado en el Paso 9.

# El log completo de conda es muy largo (incluye decenas de dependencias de
# soporte) y poco útil para reportes. Lo capturamos y, si todo sale bien,
# solo mostramos un resumen con las versiones de las herramientas pedidas
# (útil para citar versiones en informes/artículos). Si algo falla, se
# muestra el log completo para poder depurar.
import subprocess
import json

herramientas = [
    "bakta",
    "ncbi-amrfinderplus",
    "plasmidfinder",
    "integron_finder",
    "isescan",
]

def mostrar_versiones_instaladas(paquetes):
    """Imprime una tabla compacta con nombre y versión de cada paquete instalado.
    Nota: 'conda list' solo acepta UN patrón regex como filtro, por eso se
    listan todos los paquetes del entorno y se filtra en Python."""
    res = subprocess.run(["conda", "list", "--json"], capture_output=True, text=True)
    try:
        instalados = {p["name"]: p["version"] for p in json.loads(res.stdout)}
    except (json.JSONDecodeError, KeyError):
        instalados = {}
    print("\n📦 Versiones instaladas:")
    for pkg in paquetes:
        print(f"  - {pkg}: {instalados.get(pkg, '⚠️ no encontrado')}")

print("Instalando herramientas bioinformáticas (esto puede tardar varios minutos)...")
cmd = ["conda", "install", "-y", "-c", "bioconda", "-c", "conda-forge"] + herramientas
resultado = subprocess.run(cmd, capture_output=True, text=True)

if resultado.returncode == 0:
    print("✅ Herramientas bioinformáticas instaladas")
    mostrar_versiones_instaladas(herramientas)
else:
    print("❌ Falló la instalación. Log completo de conda:\n")
    print(resultado.stdout)
    print(resultado.stderr)
    raise RuntimeError("La instalación de herramientas bioinformáticas falló. Revisa el log arriba.")

Channels:
 - bioconda
 - conda-forge
Platform: linux-64
Solving environment: | / - \ | / - done

## Package Plan ##

  environment location: /usr/local

  added / updated specs:
    - antismash
    - bakta
    - integron_finder
    - isescan
    - ncbi-amrfinderplus
    - plasmidfinder


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    about-time-4.2.1           |     pyhd8ed1ab_1          16 KB  conda-forge
    alive-progress-3.0.1       |     pyhd8ed1ab_0          62 KB  conda-forge
    antismash-8.0.4            |     pyhdfd78af_1        26.7 MB  bioconda
    aragorn-1.2.41             |       h7b50bb2_5         144 KB  bioconda
    attrs-26.1.0               |     pyhcf101f3_0          63 KB  conda-forge
    bakta-1.12.0               |     pyhdfd78af_0         104 KB  bioconda
    bcbio-gff-0.7.1            |     pyhdfd78af_3          24 KB  bioconda
    biopython-1.81 

ahora vamos a instalar otras herramientas propias de **Python** que nos ayudaran a procesar y ver los resutlados mas facil para los humanos

In [3]:
!pip install matplotlib pandas --quiet

# Imports centralizados — todos en un solo lugar para evitar duplicación
import json
import os
import pathlib
import glob
import subprocess
from IPython.display import Image, display, SVG

import matplotlib.pyplot as plt
import pandas as pd

os.environ['PATH'] = '/opt/conda/bin:' + os.environ['PATH']

print("✅ Dependencias Python listas")

✅ Dependencias Python listas


## 📥 Paso 1 — Descargar los datos del caso asignado

Solo trabajaremos con un solo caso. Use el **mismo número** que trabajó en los Módulos 4 (filogenética) y 5 (ensamblaje) — asígnelo a la variable `CASO` ("01" a "20").

A diferencia del Módulo 5, aquí **no se descargan lecturas FASTQ ni se ensambla nada**: se descarga directamente el **genoma de referencia de NCBI** (el mismo usado para evaluar con QUAST en el Módulo 5) y se anota tal cual. Esto garantiza que todos los estudiantes con el mismo caso trabajen sobre un genoma idéntico y completo.

Consulte la [guía compartida](00_genome_annotation_common.md#-casos-de-estudio) para la tabla completa de casos, grupos de AMRFinderPlus y recomendación de antiSMASH.

In [4]:
# ⚠️ **IMPORTANTE**: Define el caso a trabajar aquí y solo aquí.
# Use el MISMO número de caso que trabajó en los Módulos 4 y 5 ("01" a "20").
CASO = "01"  # Por ejemplo, para el caso 1 (Klebsiella pneumoniae)

In [5]:
# Diccionario con los 20 casos: especie, genoma de referencia NCBI (el mismo usado para QUAST en el Módulo 5)
# y metadatos para las herramientas de abajo (grupo AMRFinderPlus, si antiSMASH es especialmente recomendado).
# Caso 02: el Módulo 5 usó DOS referencias (una por tecnología, cepas distintas). Para anotación se usa
# una sola: el genoma COMPLETO (GCF_030285565.1, cepa C51) en lugar del borrador de 74 contigs.
CASOS_CONFIG = {
    "01": {"especie": "Klebsiella pneumoniae",       "genoma_mb": 5.5, "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/061/393/185/GCF_061393185.1_ASM6139318v1/GCF_061393185.1_ASM6139318v1_genomic.fna.gz",       "amr_organismo": "Klebsiella_pneumoniae",     "antismash_recomendado": False},
    "02": {"especie": "Escherichia coli",             "genoma_mb": 5.0, "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/030/285/565/GCF_030285565.1_ASM3028556v1/GCF_030285565.1_ASM3028556v1_genomic.fna.gz",       "amr_organismo": "Escherichia",               "antismash_recomendado": False},
    "03": {"especie": "Pseudomonas abieticivorans",   "genoma_mb": 6.7, "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/023/509/015/GCF_023509015.1_ASM2350901v1/GCF_023509015.1_ASM2350901v1_genomic.fna.gz",       "amr_organismo": "Pseudomonas_aeruginosa",    "antismash_recomendado": True},
    "04": {"especie": "Streptomyces venezuelae",      "genoma_mb": 8.2, "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/050/632/295/GCA_050632295.1_ASM5063229v1/GCA_050632295.1_ASM5063229v1_genomic.fna.gz",       "amr_organismo": None,                         "antismash_recomendado": True},
    "05": {"especie": "Streptomyces coelicolor",      "genoma_mb": 8.7, "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/047/824/265/GCF_047824265.1_ASM4782426v1/GCF_047824265.1_ASM4782426v1_genomic.fna.gz",       "amr_organismo": None,                         "antismash_recomendado": True},
    "06": {"especie": "Bacillus subtilis",            "genoma_mb": 4.2, "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/982/518/465/GCF_982518465.1_NRS2125/GCF_982518465.1_NRS2125_genomic.fna.gz",                 "amr_organismo": None,                         "antismash_recomendado": True},
    "07": {"especie": "Staphylococcus aureus (MRSA)",  "genoma_mb": 2.8, "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/982/303/505/GCF_982303505.1_24S859_ONT_hybrid/GCF_982303505.1_24S859_ONT_hybrid_genomic.fna.gz", "amr_organismo": "Staphylococcus_aureus",     "antismash_recomendado": False},
    "08": {"especie": "Vibrio cholerae",               "genoma_mb": 4.0, "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/055/797/365/GCF_055797365.1_ASM5579736v1/GCF_055797365.1_ASM5579736v1_genomic.fna.gz",       "amr_organismo": "Vibrio_cholerae",           "antismash_recomendado": False},
    "09": {"especie": "Mycobacterium tuberculosis",    "genoma_mb": 4.4, "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/061/392/705/GCF_061392705.1_ASM6139270v1/GCF_061392705.1_ASM6139270v1_genomic.fna.gz",       "amr_organismo": "Mycobacterium_tuberculosis","antismash_recomendado": False},
    "10": {"especie": "Agrobacterium tumefaciens",     "genoma_mb": 5.6, "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/047/731/285/GCA_047731285.1_ASM4773128v1/GCA_047731285.1_ASM4773128v1_genomic.fna.gz",       "amr_organismo": None,                         "antismash_recomendado": False},
    "11": {"especie": "Deinococcus radiodurans",       "genoma_mb": 3.3, "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/045/277/105/GCF_045277105.1_KDR13939K_1.0/GCF_045277105.1_KDR13939K_1.0_genomic.fna.gz",       "amr_organismo": None,                         "antismash_recomendado": False},
    "12": {"especie": "Thermus thermophilus",          "genoma_mb": 2.1, "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/059/705/435/GCF_059705435.1_ASM5970543v1/GCF_059705435.1_ASM5970543v1_genomic.fna.gz",       "amr_organismo": None,                         "antismash_recomendado": False},
    "13": {"especie": "Lactobacillus acidophilus",     "genoma_mb": 2.0, "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/988/235/355/GCF_988235355.1_IF3HA23006/GCF_988235355.1_IF3HA23006_genomic.fna.gz",         "amr_organismo": None,                         "antismash_recomendado": False},
    "14": {"especie": "Helicobacter pylori",           "genoma_mb": 1.6, "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/059/997/335/GCA_059997335.1_ASM5999733v1/GCA_059997335.1_ASM5999733v1_genomic.fna.gz",       "amr_organismo": None,                         "antismash_recomendado": False},
    "15": {"especie": "Salmonella enterica",           "genoma_mb": 4.8, "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/061/253/545/GCF_061253545.1_ASM6125354v1/GCF_061253545.1_ASM6125354v1_genomic.fna.gz",       "amr_organismo": "Salmonella",                "antismash_recomendado": False},
    "16": {"especie": "Bacillus licheniformis",        "genoma_mb": 4.3, "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/055/397/075/GCA_055397075.1_ASM5539707v1/GCA_055397075.1_ASM5539707v1_genomic.fna.gz",       "amr_organismo": None,                         "antismash_recomendado": False},
    "17": {"especie": "Paenibacillus polymyxa",        "genoma_mb": 5.8, "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/056/645/015/GCF_056645015.1_ASM5664501v1/GCF_056645015.1_ASM5664501v1_genomic.fna.gz",       "amr_organismo": None,                         "antismash_recomendado": True},
    "18": {"especie": "Synechocystis sp.",             "genoma_mb": 3.6, "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/987/480/225/GCA_987480225.1_glgC6/GCA_987480225.1_glgC6_genomic.fna.gz",                     "amr_organismo": None,                         "antismash_recomendado": False},
    "19": {"especie": "Enterococcus faecalis",         "genoma_mb": 3.0, "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/061/255/735/GCA_061255735.1_ASM6125573v1/GCA_061255735.1_ASM6125573v1_genomic.fna.gz",       "amr_organismo": "Enterococcus_faecalis",     "antismash_recomendado": False},
    "20": {"especie": "Shewanella oneidensis",         "genoma_mb": 4.9, "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/146/165/GCF_000146165.2_ASM14616v2/GCF_000146165.2_ASM14616v2_genomic.fna.gz",         "amr_organismo": None,                         "antismash_recomendado": False},
}

if CASO not in CASOS_CONFIG:
    raise ValueError(f"⚠️ CASO '{CASO}' no reconocido. Use un número de \"01\" a \"20\".")

cfg = CASOS_CONFIG[CASO]
print(f"── Caso {CASO}: {cfg['especie']} (~{cfg['genoma_mb']} Mb) ──")

workdir = f"genome_annotattion/caso_{CASO}"
os.makedirs(f"{workdir}/data", exist_ok=True)

ref_gz = f"{workdir}/data/ref.fna.gz"
contigs_path = f"{workdir}/data/contigs.fasta"

print("Descargando genoma de referencia (el mismo usado para QUAST en el Módulo 5)...")
!wget -q "{cfg['ref_url']}" -O "{ref_gz}"
size = os.path.getsize(ref_gz) if os.path.exists(ref_gz) else 0
print(f"  Descargado: {size:,} bytes")
if size < 1000:
    raise RuntimeError(f"❌ Descarga fallida o archivo vacío: {ref_gz}")

!gunzip -f "{ref_gz}"
ref_fna = f"{workdir}/data/ref.fna"
os.rename(ref_fna, contigs_path)

# Contar contigs/secuencias
import subprocess
result = subprocess.run(["grep", "-c", ">", contigs_path], capture_output=True, text=True)
num_contigs = result.stdout.strip()
print(f"✅ Caso {CASO} listo — {num_contigs} secuencia(s) en {contigs_path}")

── CASO_C: Streptomyces venezuelae (~8.2 Mb, GC ~72%) ──
Descargando contigs comprimidos...
  Descargado: 2,287,319 bytes
Renombrado → caso_C/data/contigs.fasta
✅ Caso caso_C listo — 1 contigs en caso_C/data/contigs.fasta


## 👀 Paso 1.1 — Inspeccionar los archivos descargados

Vamos a verificar que los archivos FASTQ y el genoma de referencia FASTA se hayan descargado correctamente y tengan el formato esperado. Revisaremos el encabezado de cada uno.

In [6]:
print(f"=== Encabezado del archivo FASTA de contigs para {CASO} ===")
!head -n 10 "{contigs_path}"

print(f"\n=== Estadísticas básicas ===")
!grep -c '>' "{contigs_path}" | awk '{{print "Contigs totales: " $1}}'
!awk '/^>/{{next}}{{total+=length($0)}} END{{print "Bases totales: " total}}' "{contigs_path}"

=== Encabezado del archivo FASTA de contigs para caso_C ===
>NC_018750.1 Streptomyces venezuelae ATCC 10712, complete sequence
CCGGTCGACGAGCGGGCTTGTCCCCTGCCGGGCTGGTGCTTCTGGTGGTCGGGCCCGGCCGGGTGGTCAGGCTGAGAAGC
CGATGCTGGACACGTACTCGTACTGACCCCACTGGGAGCCGAGGTCCTGGACGACCTCACCAACCCAGATGTCGTCCATG
CCGGCATGGTCATCAGCAGCCCACGCGCCGATGATCCGGTCCCACCGACGAACGTTCAGGTGCCGGTACTCCACAACCCG
TTGGAGAGGACGTGCGACCTGGGACTGATTGAGGGGGTGGAACTCCACGCGTGTGCCGCGTCCCTGCCGGTTGAGACGGT
TCAGAAGGTACCGGGCGACGTTCTGACGCCGTACGGCACGATACGCAGACTCGATACGTTCCAAGTTCCGATGTGAAGGA
GCGCGCTTGCCCTCCCGCCAAGCTCTGATGGTCCGGTCCGTGACGGTCAGGCCAGCCGCACGCGCGGCCTGACGAGCGTG
CTCACTCCGGGTGAGATAGTTCAGCCTCGCCATCAGACCACGCCGCTGCTCAACCGGCGTCACCACAAAGCCCGCAAGCT
CATCCAGCTGACGCGCCACCAACTCATGACCCTTCACACCACGAGCCCCGAACTTCCCGAAATCAATCACCCGCTCCGGC
ACAACCCCACCCCCCATCCACCACTAAACCGCCCCCACAGCGAGAGCATGCCGTAAATACCTACACCCCCAAAATACCAG

=== Estadísticas básicas ===
grep: {contigs_path}: No such file or directory
awk: cannot open {contigs_path} (No such file or directory)


## 🏷️ Paso 2 — Anotación principal con Bakta

Bakta realiza la anotación estructural (CDS, ARNt, ARNr, ncRNA, CRISPR) y funcional en un solo paso.

> **Nota:** Bakta requiere su base de datos. La descargamos en formato light (~1 GB) — suficiente para esta práctica.

In [7]:
# Descargar base de datos de Bakta (versión light)

!mkdir -p /content/opt/bakta_db
!bakta_db download --output /content/opt/bakta_db --type light
print("✅ Base de datos Bakta lista")

Bakta software version: 1.12.0
Required database schema version: 6

Selected DB type: light

Fetch DB versions...
	... compatible DB versions: 1
Download database: v6.0, type=light, 2025-02-24, DOI: 10.5281/zenodo.14916843, URL: https://zenodo.org/record/14916843/files/db-light.tar.xz...
|████████████████████████████████████████| 1.34G/1.34G [100%] in 22:30.8 (994.63
	... done
Check MD5 sum...
	...database file OK: 4a6e059ded39e9c5537ef4137d2f5648
Extract DB tarball: file=/content/opt/bakta_db/db-light.tar.xz, output=/content/opt/bakta_db
Successfully downloaded Bakta database!
	version: 6.0
	Type: light
	DOI: 10.5281/zenodo.14916843
	path: /content/opt/bakta_db/db-light
Update AMRFinderPlus database...
	... done

Run Bakta using '--db /content/opt/bakta_db/db-light' or set a BAKTA_DB environment variable: 'export BAKTA_DB=/content/opt/bakta_db/db-light'
✅ Base de datos Bakta lista


In [8]:
# Clear MPLBACKEND environment variable to avoid matplotlib backend conflicts
if 'MPLBACKEND' in os.environ:
    del os.environ['MPLBACKEND']

# Realizar la anotación con bakta
bakta_out = f"{workdir}/results/bakta_results"
os.makedirs(bakta_out, exist_ok=True)

!bakta \
  --db /content/opt/bakta_db/db-light \
  --output "{bakta_out}" \
  --force \
  --prefix genome \
  --keep-contig-headers \
  "{contigs_path}"

print(f"✅ Bakta completo. Resultados en {bakta_out}/")

Parse genome sequences...
	imported: 1
	filtered & revised: 1
	chromosomes: 1

Start annotation...
predict tRNAs...
	found: 74
predict tmRNAs...
	found: 1
predict rRNAs...
	found: 15
predict ncRNAs...
Traceback (most recent call last):
  File "/usr/local/bin/bakta", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.11/site-packages/bakta/main.py", line 201, in main
    ncrnas = nc_rna.predict_nc_rnas(data, sequences_path)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/site-packages/bakta/features/nc_rna.py", line 40, in predict_nc_rnas
    proc = sp.run(
           ^^^^^^^
  File "/usr/local/lib/python3.11/subprocess.py", line 550, in run
    stdout, stderr = process.communicate(input, timeout=timeout)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/subprocess.py", line 1209, in communicate
    stdout, stderr = self._communicate(input, endtime, time

## 📊 Paso 3 — Explorar el resumen de Bakta con Python

In [10]:
txt_path = pathlib.Path(f"{bakta_out}/genome.txt")
print(txt_path.read_text())

FileNotFoundError: [Errno 2] No such file or directory: 'caso_C/results/bakta_results/genome.txt'

In [11]:
# Extraer métricas clave del resumen
txt = txt_path.read_text()
metricas = {}
for linea in txt.splitlines():
    if ":" in linea:
        clave, _, valor = linea.partition(":")
        metricas[clave.strip()] = valor.strip()

df_resumen = pd.DataFrame.from_dict(metricas, orient="index", columns=["Valor"])
print(df_resumen.to_string())

FileNotFoundError: [Errno 2] No such file or directory: 'caso_C/results/bakta_results/genome.txt'

### Visualización del `circular plot` generado por Bakta

In [ ]:
print(f"Archivos en {bakta_out}:")
!ls "{bakta_out}"

circular_plot_path = os.path.join(bakta_out, "genome.svg")

if os.path.exists(circular_plot_path):
    print(f"\nMostrando el circular plot: {circular_plot_path}")
    display(SVG(filename=circular_plot_path))
else:
    print("\nNo se encontró 'genome.svg'. Verifique que Bakta haya completado correctamente.")

**Preguntas:**

1. ¿Cuántas secuencias/contigs había en el genoma de referencia usado?
2. ¿Cuál es la longitud total del genoma?
3. ¿Cuántos CDS fueron predichos?
4. ¿Cuántas *small proteins* se encontraron?
5. ¿Cuántos ARNt y ARNr se anotaron?
6. ¿Hay secuencias CRISPR?
7. Compare el número de CDS reportado por Bakta con la anotación original de NCBI/RefSeq para su accesión (columna "Referencia" en la [tabla de casos](00_genome_annotation_common.md#-casos-de-estudio)). ¿Coinciden aproximadamente?

## 🦠 Paso 4 — Genes de resistencia con AMRFinderPlus

In [ ]:
# El grupo taxonómico se toma automáticamente de CASOS_CONFIG (None si el caso no tiene grupo disponible en AMRFinderPlus)
ORGANISMO = cfg["amr_organismo"]
print(f"Grupo taxonómico AMRFinderPlus: {ORGANISMO if ORGANISMO else '(ninguno — búsqueda genérica)'}")

# Actualizar base de datos AMRFinder
!amrfinder -u
print("✅ Base de datos AMRFinder actualizada")

amr_out = f"{workdir}/results/amrfinder_results"
os.makedirs(amr_out, exist_ok=True)
amr_tsv = f"{amr_out}/amrfinder.tsv"

organism_flag = f"--organism {ORGANISMO}" if ORGANISMO else ""

!amrfinder \
  --nucleotide "{contigs_path}" \
  -o "{amr_tsv}" \
  {organism_flag} \
  --threads 2

print("✅ AMRFinderPlus completo")
print(f"\n--- Primeras 5 líneas de {amr_tsv} ---")
!head -n 5 "{amr_tsv}"

In [ ]:
amr_df = pd.read_csv(amr_tsv, sep="\t")
print(f"Total de genes encontrados: {len(amr_df)}\n")

# FIX: verificar que las columnas existen antes de seleccionarlas
cols_disponibles = [c for c in ["Element symbol", "Contig id", "Class", "Subclass", "% Identity to reference"] if c in amr_df.columns]
print(amr_df[cols_disponibles].to_string())

**Preguntas:**

8. ¿Qué genes de resistencia a antibióticos se encontraron?
9. ¿A qué familias de antibióticos pertenecen (columna `Class`)?
10. ¿En qué contigs se encuentran?
11. ¿Se encontraron factores de virulencia?

## 🔵 Paso 5 — Identificación de plásmidos con PlasmidFinder

In [ ]:
pf_out = f"{workdir}/results/plasmidfinder_results"
os.makedirs(pf_out, exist_ok=True)

# la ruta de la base de datos de PlasmidFinder varía según la versión instalada.
# Se detecta dinámicamente en lugar de usar una ruta fija hardcodeada.
pf_db_candidates = glob.glob("/usr/local/share/plasmidfinder*/database") + \
                   glob.glob("/opt/conda/share/plasmidfinder*/database")

if not pf_db_candidates:
    raise RuntimeError("❌ No se encontró la base de datos de PlasmidFinder. Verifique la instalación.")

pf_db = pf_db_candidates[0]
print(f"Usando base de datos: {pf_db}")

!plasmidfinder.py \
  -i "{contigs_path}" \
  -o "{pf_out}" \
  -p "{pf_db}" \
  -x

print("✅ PlasmidFinder completo")

In [ ]:
pf_tsv = f"{pf_out}/results_tab.tsv"
pf_df = pd.read_csv(pf_tsv, sep="\t")
print(f"Plásmidos encontrados: {len(pf_df)}\n")

# FIX: verificar columnas disponibles antes de seleccionarlas
cols_pf = [c for c in ["Plasmid", "Identity", "Query / Template length", "Contig", "Position in contig"] if c in pf_df.columns]
print(pf_df[cols_pf].to_string())

**Preguntas:**

12. ¿Cuántos plásmidos se identificaron?
13. ¿A qué replicones / familias pertenecen?
14. ¿En qué contigs se encuentran?
15. Cruzando con AMRFinderPlus: ¿algún gen de resistencia está en el mismo contig que un plásmido?

## 🔗 Paso 6 — Detección de integrones con IntegronFinder

In [ ]:
integron_out = f"{workdir}/results/integron_results"
os.makedirs(integron_out, exist_ok=True)

!integron_finder \
  --local-max \
  --func-annot "{contigs_path}" \
  --outdir "{integron_out}"

print("✅ IntegronFinder completo")

In [ ]:
summary_files = list(pathlib.Path(integron_out).glob("*.summary"))
if summary_files:
    print(summary_files[0].read_text())
else:
    print("⚠️ No se encontró archivo .summary. Verifique que IntegronFinder haya completado correctamente.")
    print(f"Archivos en {integron_out}:")
    !ls "{integron_out}"

**Preguntas:**

16. ¿Cuántos integrones completos se encontraron?
17. ¿Cuántos elementos In0 y CALIN?
18. ¿En qué contigs se localizan?

## 🔀 Paso 7 — Detección de elementos IS con ISEScan

In [ ]:
isescan_out = f"{workdir}/results/isescan_results"

!isescan.py \
  --seqfile "{contigs_path}" \
  --output "{isescan_out}" \
  --nthread 4

print("✅ ISEScan completo")

In [ ]:
is_files = glob.glob(f"{isescan_out}/data/*.tsv")
if is_files:
    is_df = pd.read_csv(is_files[0], sep="\t")
    print(f"Elementos IS detectados: {len(is_df)}\n")
    # FIX: verificar columnas disponibles (los nombres pueden variar entre versiones de ISEScan)
    cols_is = [c for c in ["family", "cluster", "seqID", "isBegin", "isEnd"] if c in is_df.columns]
    print(is_df[cols_is].to_string())
else:
    print("No se encontraron elementos IS o el archivo aún no está listo.")

**Preguntas:**

19. ¿Cuántos elementos IS se detectaron?
20. ¿En qué contigs se encuentran?
21. ¿Cuáles son las distintas familias de IS identificadas?

## 📊 Paso 8 — Resumen integrado con Python

Construye una tabla resumen que cruce los resultados de Bakta, AMRFinderPlus y PlasmidFinder por contig.

In [ ]:
# Cargar resultados
amr_df  = pd.read_csv(amr_tsv, sep="\t")
pf_df   = pd.read_csv(pf_tsv,  sep="\t")

# Contigs con genes de resistencia
amr_col = "Contig id" if "Contig id" in amr_df.columns else amr_df.columns[1]
pf_col  = "Contig"    if "Contig"    in pf_df.columns  else pf_df.columns[1]

amr_contigs     = set(amr_df[amr_col].dropna().unique())
plasmid_contigs = set(pf_df[pf_col].dropna().unique())
both            = amr_contigs & plasmid_contigs

print("=== Contigs con genes de resistencia ===")
print(sorted(amr_contigs) if amr_contigs else "Ninguno")
print("\n=== Contigs con plásmidos ===")
print(sorted(plasmid_contigs) if plasmid_contigs else "Ninguno")
print("\n=== Contigs con AMBOS (resistencia + plásmido) ===")
print(sorted(both) if both else "Ninguno")

## 🌿 Paso 9 — Predicción de BGC con antiSMASH

> **Este paso se puede ejecutar para cualquier caso**, pero es especialmente relevante si su caso está marcado como antiSMASH recomendado (✅) en la [tabla de casos](00_genome_annotation_common.md#-casos-de-estudio) — géneros con historial conocido de producción de metabolitos secundarios (*Streptomyces*, *Pseudomonas*, *Bacillus*, *Paenibacillus*, entre otros).

Los clústeres de genes biosintéticos (BGC) son regiones del genoma que codifican la maquinaria completa para sintetizar un metabolito secundario — antibióticos, antifúngicos, sideróforos, pigmentos, etc. **antiSMASH** es la herramienta estándar para su predicción.

In [ ]:
print(f"Caso {CASO} ({cfg['especie']}): antiSMASH recomendado = {cfg['antismash_recomendado']}")

Si su caso **no** está marcado como recomendado, igual puede ejecutar este paso por curiosidad — simplemente es menos probable encontrar BGC interesantes, pero no está prohibido.

> ⚠️ antiSMASH no se instala con el bloque `conda install` anterior porque tiene dependencias que pueden entrar en conflicto. Se instala aquí de forma independiente.

In [ ]:
# Instalar antiSMASH en un entorno conda SEPARADO (puede tardar 5–10 minutos)
#
# Por qué un entorno aparte: Google Colab ahora corre Python 3.13, y
# condacolab fija (pin) esa versión en el entorno 'base'. antiSMASH (tanto
# 8.x como 7.1.0) depende de versiones exactas de biopython que requieren
# Python <=3.12, así que NO puede instalarse en el entorno base pineado a
# 3.13 — da igual qué versión de antiSMASH se intente. La solución es crear
# un entorno conda independiente (antismash_env) con su propio Python, y
# ejecutar antiSMASH ahí vía 'conda run -n antismash_env ...'. Esto no
# afecta al kernel de Jupyter (que sigue usando el entorno base).
#
# Intentamos primero con la versión más reciente de antiSMASH. Si el solver
# de conda falla, reintentamos fijando antiSMASH a la versión 7.1.0, que se
# sabe compatible. El log completo de conda solo se muestra si algo falla;
# si todo va bien, solo se imprime la versión instalada (útil para citar en
# informes/artículos).
import subprocess
import json

ANTISMASH_ENV = "antismash_env"

def antismash_env_existe():
    res = subprocess.run(["conda", "env", "list", "--json"], capture_output=True, text=True)
    try:
        envs = json.loads(res.stdout).get("envs", [])
    except json.JSONDecodeError:
        envs = []
    return any(e.rstrip("/").endswith(ANTISMASH_ENV) for e in envs)

def crear_env_antismash(spec):
    cmd = ["conda", "create", "-y", "-n", ANTISMASH_ENV, "-c", "bioconda", "-c", "conda-forge", spec]
    print(f"➡️  Creando entorno '{ANTISMASH_ENV}' con {spec}...")
    return subprocess.run(cmd, capture_output=True, text=True)

if antismash_env_existe():
    print(f"ℹ️  El entorno '{ANTISMASH_ENV}' ya existe, se omite la instalación.")
    resultado = None
else:
    print("Intentando instalar la versión más reciente de antiSMASH...")
    resultado = crear_env_antismash("antismash")

    if resultado.returncode != 0:
        print("⚠️  Falló la instalación con antiSMASH (última versión). Reintentando con antiSMASH 7.1.0 (versión compatible conocida)...")
        resultado = crear_env_antismash("antismash=7.1.0")

if resultado is None or resultado.returncode == 0:
    res = subprocess.run(["conda", "list", "-n", ANTISMASH_ENV, "--json"], capture_output=True, text=True)
    try:
        version = next(p["version"] for p in json.loads(res.stdout) if p["name"] == "antismash")
    except (json.JSONDecodeError, StopIteration):
        version = "desconocida"
    print(f"✅ antiSMASH instalado en el entorno '{ANTISMASH_ENV}' (versión {version})")
else:
    print("❌ Falló la instalación. Log completo de conda:\n")
    print(resultado.stdout)
    print(resultado.stderr)
    raise RuntimeError("No se pudo instalar antiSMASH (ni la versión actual ni 7.1.0). Revisa el log arriba.")

In [ ]:
# Descargar base de datos de antiSMASH
!mkdir -p /content/opt/antismash_db
!download-antismash-databases --database-dir /content/opt/antismash_db
print("✅ Base de datos antiSMASH lista")

In [ ]:
antismash_out = f"{workdir}/results/antismash_results"
os.makedirs(antismash_out, exist_ok=True)

!antismash \
  "{contigs_path}" \
  --output-dir "{antismash_out}" \
  --taxon bacteria \
  --genefinding-tool prodigal \
  --pfam2go \
  --databases /content/opt/antismash_db \
  --cpus 4

print(f"✅ antiSMASH completo. Reporte en {antismash_out}/index.html")

### Explorar los resultados de antiSMASH con Python

In [ ]:
import re # Added import for regular expressions

json_files = list(pathlib.Path(antismash_out).glob("*.json"))
if not json_files:
    raise FileNotFoundError(f"No se encontró archivo JSON en {antismash_out}. ¿Completó antiSMASH?")

data = json.loads(json_files[0].read_text())

# Extraer información de todos los BGC encontrados
registros = []
for record in data.get("records", []):
    for feature in record.get("features", []):
        if feature.get("type") == "region":
            quals = feature.get("qualifiers", {})

            # Parse the location string
            location_str = feature["location"]
            match = re.match(r'\[(\d+):(\d+)\]\(.+\)', location_str)
            if match:
                start_coord = int(match.group(1))
                end_coord = int(match.group(2))
            else:
                start_coord = 0
                end_coord = 0 # Default values if parsing fails

            registros.append({
                "Contig":          record["id"],
                "BGC_type":        ", ".join(quals.get("product", ["desconocido"])),
                "Start":           start_coord,
                "End":             end_coord,
                "Longitud_pb":     end_coord - start_coord,
                "KnownCluster_hit": quals.get("knownclusterblast", ["—"])[0][:60]
                                    if quals.get("knownclusterblast") else "—"
            })

bgc_df = pd.DataFrame(registros)
print(f"Total de BGC predichos: {len(bgc_df)}\n")
print(bgc_df.to_string())

In [ ]:
# Distribución de tipos de BGC
tipo_counts = bgc_df["BGC_type"].value_counts()

fig, ax = plt.subplots(figsize=(8, 10))
tipo_counts.plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("Número de BGC")
ax.set_title(f"Tipos de BGC predichos — {CASO}\n(antiSMASH, n={len(bgc_df)})")
ax.invert_yaxis()
plt.tight_layout()

bgc_plot = f"{antismash_out}/bgc_tipos_{CASO}.png"
plt.savefig(bgc_plot, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Gráfico guardado en {bgc_plot}")

**Preguntas — antiSMASH:**

26. ¿Cuántos BGC predijo antiSMASH en el genoma de su caso?
27. ¿Qué tipos de BGC están presentes? ¿Cuál es el más frecuente?
28. ¿Alguno de los BGC tiene homología con un clúster conocido según KnownClusterBlast (ej. un antibiótico, sideróforo o metabolito reportado en la literatura para su especie)?
29. ¿Hay BGC sin homología conocida en MIBiG? ¿Cuántos? ¿Qué implicaría caracterizar uno de ellos experimentalmente?
30. Elija un BGC que le llame la atención y describa: tipo, tamaño en pb y metabolito predicho.
31. Si su caso **no** tenía antiSMASH recomendado pero igual lo ejecutó: ¿encontró BGC? ¿Cómo se compara en número/diversidad con un caso que sí esté marcado como recomendado (puede comparar con un compañero)?

## 📦 Paso 10 — Comprimir todos los resultados

In [ ]:
all_results_archive = f"{workdir}/results/all_results.zip"

# Comprimir toda la carpeta de resultados
!zip -r "{all_results_archive}" "{workdir}/results"

print(f"\n✅ Todos los resultados comprimidos en: {all_results_archive}")
print("Puedes descargar este archivo desde el explorador de archivos de Colab.")

## ❓ Preguntas integradoras

**Todos los casos:**

32. Considerando los resultados de Bakta, AMRFinderPlus y PlasmidFinder: ¿qué puede concluir sobre la movilidad de los genes de resistencia en su caso?
33. Si un gen de resistencia está en un plásmido que también contiene un integrón, ¿qué implica para la diseminación de resistencias?
34. ¿Todos los contigs con plásmidos pertenecen claramente al organismo de estudio? ¿Cómo lo verificaría?
35. Si su caso tiene grupo taxonómico en AMRFinderPlus (ver tabla de casos): ¿los resultados son consistentes con lo que esperaría para esa especie/cepa según la guía del Módulo 5?

**Si ejecutó antiSMASH (Paso 9):**

36. ¿Cuántos BGC identificó Bakta en su anotación estándar (sección de la salida relacionada con regiones/clusters)? ¿Coincide con los predichos por antiSMASH? ¿Por qué podrían diferir?
37. Compare el número de BGC predichos con lo reportado en la literatura para su especie (si existe). ¿Cuántos tienen producto conocido y cuántos son "silenciosos" o sin homología en MIBiG?
38. ¿Qué ventaja tiene usar un genoma *finished* (como el GCF_*/GCA_*.1 de referencia usado aquí) en lugar de un borrador (*draft*) con muchos contigs para este tipo de análisis?

---
*Consulte la [guía compartida](00_genome_annotation_common.md) y el [README del Módulo 6](../README.md) para repasar los conceptos.*